# Lesson 5: Metrics and statistics (just enough)

Companion to `curriculum/05-metrics-and-statistics.md`.

**Big idea:** every eval number has an error bar. With small datasets and nondeterministic models, many
"improvements" are noise.

Mock model only: free, offline, no API key. Run cells top to bottom.
**🧠 Check** = answer in your own words before moving on. **✍️ Your turn** = cells for you to write (ask Claude for a hint).

> Your thread through the course: an AI **validator** that reads an Issue from the GRC platform and checks
> its rating against the Incident & Issue Management Standard.

In [ ]:
import json, random, re
from collections import Counter, defaultdict

from evalkit import app, graders
from evalkit.stats import bootstrap_ci, mean, paired_bootstrap_delta, wilson, cohens_kappa
from nb_helpers import DEV, HOLDOUT, load, run, report, failures, mock_task

dev = load(DEV)
holdout = load(HOLDOUT)
print(f"dev={len(dev)} cases, holdout={len(holdout)} cases")

## 1. Two sources of noise

1. **Sampling noise (which cases):** with 20 cases, one flip moves the pass rate by 5 points.
2. **Model nondeterminism (which run):** the same input gives different outputs. Let's measure it with 5 trials.

In [ ]:
recs5 = run(dev, trials=5)
s5 = report(recs5, trials=5, label="5 trials")

print("\ncases whose priority changed across trials:")
by_case = defaultdict(list)
for r in recs5:
    by_case[r["case_id"]].append(r["output"]["priority"])
for cid, ps in by_case.items():
    if len(set(ps)) > 1:
        print(f"  {cid}: {ps}")

## 2. pass@k vs pass^k

- **pass@k:** passed *at least once* in k tries. Relevant when something picks the best attempt.
- **pass^k:** passed *every* time. **This is what a user (or an auditor re-running the check) experiences.**

The gap between them measures **unreliability**.

In [ ]:
print(f"pass@5 = {s5['pass_at_k']:.0%}   pass^5 = {s5['pass_hat_k']:.0%}   gap = {s5['pass_at_k'] - s5['pass_hat_k']:.0%}")

### 🧠 Check
Your validator gives a verdict on an Issue. An auditor re-runs it a week later and gets a *different*
verdict. Which metric captures that risk, and what would you set as the minimum?

## 3. Confidence intervals

The bootstrap resamples cases with replacement thousands of times and takes the middle 95% of the
recomputed means (`evalkit/stats.py → bootstrap_ci`). `wilson()` is a formula-based alternative for pass rates.

In [ ]:
print(" n    80% pass rate: bootstrap CI     wilson CI")
for n in [10, 20, 50, 100, 400]:
    k = int(0.8 * n)
    lo, hi = bootstrap_ci([1]*k + [0]*(n-k))
    wlo, whi = wilson(k, n)
    print(f"{n:>3}   [{lo:.0%}, {hi:.0%}]            [{wlo:.0%}, {whi:.0%}]")

## 4. Paired comparison: your best friend

When comparing version A with B **on the same cases**, compare *per-case differences*. Case difficulty cancels
out, so real changes show up with far fewer cases.

Synthetic demo: 40 cases with very different difficulty; B is genuinely better by about 8 points.

In [ ]:
rng = random.Random(7)
difficulty = [rng.random() for _ in range(40)]           # per-case pass probability for A
a = [1 if rng.random() < p else 0 for p in difficulty]
b = [1 if (x == 1 or rng.random() < 0.2) else 0 for x in a]  # B fixes ~20% of A's failures, breaks none

# Unpaired: compare two separate CIs
print(f"A: {mean(a):.0%} CI{tuple(round(x, 2) for x in bootstrap_ci(a))}")
print(f"B: {mean(b):.0%} CI{tuple(round(x, 2) for x in bootstrap_ci(b))}   <- CIs overlap, looks inconclusive")

# Paired: CI on the per-case delta
d, (lo, hi) = paired_bootstrap_delta(a, b)
print(f"\npaired delta: {d:+.0%}  CI [{lo:+.0%}, {hi:+.0%}]  -> {'real' if lo > 0 else 'noise'}")
print(f"fixed {sum(y > x for x, y in zip(a, b))}, broke {sum(y < x for x, y in zip(a, b))}")

### 🧠 Check
In your own words: why does pairing shrink the interval? Also, why should you count **fixed vs broken**
cases as well as the net delta?

## 5. Choosing metrics: when classes matter differently

Overall accuracy hides what matters most. For triage, **missing an urgent ticket** is worst. That's
**recall** on `urgent`: of the truly urgent tickets, how many did we flag as urgent?

In [ ]:
recs = run(dev + holdout)
for r in recs:
    exp = r["expected"]["priority"]
    got = r["output"]["priority"]
    print(f"{r['case_id']}: expected={exp:<7} got={got}") if exp == "urgent" or got == "urgent" else None

### ✍️ Your turn
Using `recs`, compute for `urgent`:
- **recall** = (expected urgent AND predicted urgent) / (expected urgent)
- **precision** = (expected urgent AND predicted urgent) / (predicted urgent)

Then: for your validator, which matters more on the top rating level, recall or precision? Why?

In [ ]:
# ✍️ Your code here

**Also track:**
- **Guardrail metrics**: things that must stay at about 100% (policy, schema validity, "never under-rate by 2+"). Report them separately.
- **Cost & latency** next to quality.
- **Saturation**: at about 98% the eval can't tell good from better, so you need harder cases.

---
## As a lead engineer…

- **Demand error bars:** "v3 is better" should come with *n, delta, CI, and which cases regressed*.
- **Pre-register the decision rule:** decide *before* the run what result would make you ship, so nobody re-runs until it looks good.
- **No single number:** use a small dashboard of overall, guardrails, per-slice, cost, and latency.

### 🧠 Discussion
Write the pre-registered ship rule for validator v1 in one sentence (e.g. "≥ X% agreement on N confirmed
Issues, kappa ≥ Y, zero 2-level under-ratings, pass^3 ≥ Z%"). Which numbers would you need risk to sign off on?

**Recap**
- Noise comes from which cases and which run. Use CIs and multiple trials.
- Paired comparisons + regression counts beat comparing two averages.
- Pick metrics that match the cost of errors: recall on the top level, guardrails as hard floors.

**Next:** `/lab 4`, then `/lesson 6`.